---
## 📅 PARTIE 11 — Décomposition Mensuelle de l'Indice par Moughataa et Global

### Objectif
Passer de l'analyse **annuelle** à l'analyse **mensuelle** pour :
- Détecter une **saisonnalité** dans le signal SAR (saison des pluies juil-sept vs saison sèche)
- Valider que la **tendance long-terme** reste robuste après décomposition saisonnière
- Identifier quelles moughataat réagissent le plus aux variations climatiques saisonnières

### Approche
On travaille sur le **VV moyen continu (dB)** — pas de seuillage binaire à -14 dB.  
Avec seulement 2-3 images par mois, le seuillage est trop instable.  
La valeur VV moyenne continue absorbe mieux le bruit inter-image.

### Hypothèses
| # | Hypothèse | Attendu |
|---|-----------|---------|
| H1 | Saisonnalité positive juil-sept | +0.5 à +1.5 dB zones périphériques |
| H2 | Zones très urbanisées = faible saisonnalité | Ksar, Arafat < Toujounine, Riyadh |
| H3 | Tendance long-terme visible après décomposition | Valide l'indice annuel |
| H4 | Résiduel dominé par bruit SAR (2-3 img/mois) | Résiduel ±1 dB aléatoire |

### 📥 Cellule A — Extraction VV Mensuel 2016-2024
> ⏱️ **Attention : cette cellule prend ~20-30 minutes.**  
> Elle fait 972 appels GEE (9 zones × 12 mois × 9 ans).  
> Lancer et ne pas interrompre. Le résultat est sauvegardé en CSV.

In [ ]:
# ── CELL 11A — Extraction VV Mensuel par Moughataa (2016-2024) ─────────────
import calendar
import time

YEARS_MONTHLY  = list(range(2016, 2025))
MONTHS         = list(range(1, 13))
MONTH_NAMES    = ['Jan','Fév','Mar','Avr','Mai','Jun',
                  'Jul','Aoû','Sep','Oct','Nov','Déc']

print('Extraction VV mensuel par moughataa — 972 appels GEE')
print('Chaque point = 1 moughataa × 1 mois × 1 année')
print('='*60)

records_monthly = []
total           = len(YEARS_MONTHLY) * len(MONTHS) * len(gdf)
done            = 0
t0              = time.time()

for year in YEARS_MONTHLY:
    for month in MONTHS:
        last_day = calendar.monthrange(year, month)[1]
        d1       = f'{year}-{month:02d}-01'
        d2       = f'{year}-{month:02d}-{last_day}'

        col_m = s1_col.filterDate(d1, d2).select('VV')
        n_img = col_m.size().getInfo()

        if n_img == 0:
            # Pas d'image ce mois — on enregistre NaN pour toutes les zones
            for _, row in gdf.iterrows():
                records_monthly.append({
                    'year': year, 'month': month,
                    'zone': row['ADM2_EN'],
                    'vv_mean': None, 'n_images': 0
                })
            done += len(gdf)
            continue

        comp_m = col_m.median()

        for _, row in gdf.iterrows():
            zone    = row['ADM2_EN']
            geom_ee = ee.Geometry(
                _json.loads(gpd.GeoSeries([row.geometry], crs='EPSG:4326').to_json())
                ['features'][0]['geometry']
            )
            vv = comp_m.reduceRegion(
                reducer    = ee.Reducer.mean(),
                geometry   = geom_ee,
                scale      = 10,
                maxPixels  = 1e10,
                bestEffort = True
            ).getInfo().get('VV', None)

            records_monthly.append({
                'year'    : year,
                'month'   : month,
                'zone'    : zone,
                'vv_mean' : round(float(vv), 3) if vv is not None else None,
                'n_images': n_img
            })
            done += 1

        # Progress toutes les 50 zones
        if done % 50 == 0:
            elapsed = time.time() - t0
            pct     = 100 * done / total
            eta     = (elapsed / done) * (total - done) / 60
            print(f'  {done:4d}/{total}  ({pct:.0f}%)  —  {year}-{month:02d}  '
                  f'n_img={n_img}  ETA: {eta:.0f} min restantes')

df_monthly = pd.DataFrame(records_monthly)
df_monthly.to_csv('/outputs/monthly_sar_vv.csv', index=False)

print()
print(f'✓ Extraction terminée en {(time.time()-t0)/60:.1f} minutes')
print(f'  {len(df_monthly)} enregistrements')
print(f'  NaN : {df_monthly["vv_mean"].isna().sum()} ({df_monthly["vv_mean"].isna().mean()*100:.1f}%)')
print('✓ Saved: /outputs/monthly_sar_vv.csv')

### 🔍 Cellule B — Qualité des Données Mensuelles
Avant toute analyse : vérifier la couverture temporelle, le nombre d'images, les NaN.

In [ ]:
# ── CELL 11B — Qualité et Couverture des Données Mensuelles ────────────────
# Si tu relances après avoir fermé le notebook :
# df_monthly = pd.read_csv('/outputs/monthly_sar_vv.csv')

print('── Statistiques de couverture ──────────────────────────────────────────')
print(f'  Enregistrements totaux : {len(df_monthly)}')
print(f'  Zones                  : {df_monthly["zone"].nunique()}')
print(f'  Mois-années            : {df_monthly.groupby(["year","month"]).ngroups}')
print(f'  NaN (0 image ce mois)  : {df_monthly["vv_mean"].isna().sum()}')
print()

# Images par mois (moyenne sur toutes les zones)
img_per_month = (df_monthly.groupby(['year','month'])['n_images']
                 .first().unstack('month'))
print('── Nombre d\'images Sentinel-1 par mois ───────────────────────────────')
print(f'{"":6s}', end='')
for m in MONTHS: print(f'{MONTH_NAMES[m-1]:>6s}', end='')
print()
print('-'*(6+6*12))
for year in YEARS_MONTHLY:
    print(f'{year}  ', end='')
    for month in MONTHS:
        n = img_per_month.loc[year, month] if (year in img_per_month.index and month in img_per_month.columns) else 0
        print(f'{int(n):>6d}', end='')
    print()

print()
print(f'  Moyenne images/mois : {df_monthly.groupby(["year","month"])["n_images"].first().mean():.1f}')
print(f'  Min                 : {df_monthly.groupby(["year","month"])["n_images"].first().min()}')
print(f'  Mois avec 0 image   : {(df_monthly.groupby(["year","month"])["n_images"].first()==0).sum()}')
print()

# Plage VV par zone
print('── Plage VV moyen par moughataa ─────────────────────────────────────')
print(f'{"Zone":20s} {"Min":>8s} {"Max":>8s} {"Moy":>8s} {"Std":>8s}')
print('-'*55)
for zone in ZONE_ORDER:
    sub = df_monthly[df_monthly['zone']==zone]['vv_mean'].dropna()
    print(f'{zone:20s} {sub.min():>8.2f} {sub.max():>8.2f} {sub.mean():>8.2f} {sub.std():>8.2f}')

### 🗓️ Cellule C — Heatmap Brute Année × Mois
Première visualisation : voir la structure temporelle globale du signal VV.

In [ ]:
# ── CELL 11C — Heatmap VV Mensuel par Moughataa ─────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('VV Backscatter Mensuel (dB) — Nouakchott 2016-2024\n'
             'Heatmap Année × Mois par Moughataa  |  Blanc=fort=urbain  |  Bleu=faible=sable',
             fontsize=14, fontweight='bold')

cmap_vv = LinearSegmentedColormap.from_list('vv_heat',
    ['#2c7bb6','#abd9e9','#ffffbf','#fdae61','#d7191c'])

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub = df_monthly[df_monthly['zone']==zone].copy()
    pivot = sub.pivot(index='year', columns='month', values='vv_mean')

    # Valeurs min/max propres à la zone pour contraste max
    vmin = pivot.values[~np.isnan(pivot.values)].min()
    vmax = pivot.values[~np.isnan(pivot.values)].max()

    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap_vv,
                   vmin=vmin, vmax=vmax,
                   interpolation='nearest')

    # Annotations valeurs
    for i, year in enumerate(YEARS_MONTHLY):
        for j, month in enumerate(MONTHS):
            val = pivot.loc[year, month] if (year in pivot.index and month in pivot.columns) else np.nan
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                        fontsize=5.5, color='black' if (val-vmin)/(vmax-vmin+1e-10)>0.4 else 'white')

    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTH_NAMES, fontsize=7, rotation=45)
    ax.set_yticks(range(len(YEARS_MONTHLY)))
    ax.set_yticklabels([str(y) for y in YEARS_MONTHLY], fontsize=7)
    ax.set_title(zone, fontsize=10, fontweight='bold', color=ZONE_COLORS[zone])
    plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label='VV (dB)')

plt.tight_layout()
plt.savefig('/outputs/monthly_01_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_01_heatmap.png')

# Heatmap ville (moyenne pondérée superficie)
fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle('VV Backscatter Mensuel — Indice Ville Global (pondéré superficie)\n'
             'Nouakchott 2016-2024', fontsize=13, fontweight='bold')

# Calculer la moyenne pondérée ville par mois-année
city_monthly = []
for year in YEARS_MONTHLY:
    for month in MONTHS:
        sub = df_monthly[(df_monthly['year']==year) & (df_monthly['month']==month)]
        vv_w = sum(sub[sub['zone']==z]['vv_mean'].values[0] * areas[z]
                   for z in ZONE_ORDER
                   if len(sub[sub['zone']==z])>0 and sub[sub['zone']==z]['vv_mean'].values[0] is not None
                   and not np.isnan(sub[sub['zone']==z]['vv_mean'].values[0]))
        w_total = sum(areas[z] for z in ZONE_ORDER
                      if len(sub[sub['zone']==z])>0 and sub[sub['zone']==z]['vv_mean'].values[0] is not None
                      and not np.isnan(sub[sub['zone']==z]['vv_mean'].values[0]))
        city_monthly.append({'year':year,'month':month,
                             'vv_city': round(vv_w/w_total,3) if w_total>0 else np.nan})

df_city_monthly = pd.DataFrame(city_monthly)
pivot_city = df_city_monthly.pivot(index='year', columns='month', values='vv_city')

vmin_c = pivot_city.values[~np.isnan(pivot_city.values)].min()
vmax_c = pivot_city.values[~np.isnan(pivot_city.values)].max()

im = ax.imshow(pivot_city.values, aspect='auto', cmap=cmap_vv,
               vmin=vmin_c, vmax=vmax_c)
for i, year in enumerate(YEARS_MONTHLY):
    for j, month in enumerate(MONTHS):
        val = pivot_city.loc[year, month] if year in pivot_city.index else np.nan
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='black' if (val-vmin_c)/(vmax_c-vmin_c+1e-10)>0.4 else 'white')
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES, fontsize=9)
ax.set_yticks(range(len(YEARS_MONTHLY)))
ax.set_yticklabels([str(y) for y in YEARS_MONTHLY], fontsize=9)
ax.set_title('Ville de Nouakchott — Indice Global', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.6, label='VV moyen pondéré (dB)')
plt.tight_layout()
plt.savefig('/outputs/monthly_01b_heatmap_city.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_01b_heatmap_city.png')

### 🗺️ Cellule D — Cartes SAR Saison Sèche vs Saison des Pluies
Visualisation spatiale de la différence entre les deux saisons.

In [ ]:
# ── CELL 11D — Cartes SAR : Saison Sèche vs Saison des Pluies ──────────────
# Composite de TOUS les mois de janvier (saison sèche) vs TOUS les août (pluies)
# sur la période 2016-2024 → composite stable malgré peu d'images/mois

print('Construction composites saisonniers multi-annuels...')

# Tous les janvier 2016-2024 → composite saison sèche stable
col_jan = s1_col.filter(ee.Filter.calendarRange(1, 1, 'month')).select('VV')
col_aug = s1_col.filter(ee.Filter.calendarRange(8, 8, 'month')).select('VV')
col_sep = s1_col.filter(ee.Filter.calendarRange(9, 9, 'month')).select('VV')
col_dry = s1_col.filter(  # saison sèche complète (nov-avril)
    ee.Filter.Or(
        ee.Filter.calendarRange(11, 12, 'month'),
        ee.Filter.calendarRange(1,  4,  'month')
    )
).select('VV')
col_wet = s1_col.filter(  # saison des pluies (juil-oct)
    ee.Filter.calendarRange(7, 10, 'month')
).select('VV')

n_jan = col_jan.size().getInfo()
n_aug = col_aug.size().getInfo()
n_dry = col_dry.size().getInfo()
n_wet = col_wet.size().getInfo()
print(f'  Janvier (multi-an) : {n_jan} images')
print(f'  Août    (multi-an) : {n_aug} images')
print(f'  Saison sèche totale: {n_dry} images')
print(f'  Saison pluies total: {n_wet} images')

comp_jan = col_jan.median()
comp_aug = col_aug.median()
comp_dry = col_dry.median()
comp_wet = col_wet.median()

# Différence pluies - sèche
vv_diff_seasonal = comp_wet.subtract(comp_dry).rename('vv_diff')

print('Téléchargement thumbnails saisonniers...')
vv_dry_thumb  = get_colored_thumb(comp_dry, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_wet_thumb  = get_colored_thumb(comp_wet, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_jan_thumb  = get_colored_thumb(comp_jan, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_aug_thumb  = get_colored_thumb(comp_aug, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_diff_thumb = get_colored_thumb(vv_diff_seasonal, 'vv_diff', region_ee,
                    ['2166ac','4393c3','f7f7f7','f4a582','d6604d'], -2, 2)
print('✓ OK')

fig, axes = plt.subplots(1, 5, figsize=(30, 9))
fig.suptitle('Comparaison Saisonnière SAR — Saison Sèche vs Saison des Pluies\n'
             'Composites multi-annuels 2016-2024  |  Sentinel-1 VV (dB)',
             fontsize=13, fontweight='bold')

for ax, (img, title) in zip(axes, [
    (vv_jan_thumb,  'Composite Janvier\n(saison sèche — tous les jan 2016-2024)'),
    (vv_dry_thumb,  'Composite Saison Sèche\n(Nov-Avr, multi-annuel)'),
    (vv_aug_thumb,  'Composite Août\n(saison pluies — tous les aoû 2016-2024)'),
    (vv_wet_thumb,  'Composite Saison Pluies\n(Jul-Oct, multi-annuel)'),
    (vv_diff_thumb, 'ΔVV = Saison Pluies − Sèche\nRouge=hausse | Bleu=baisse'),
]):
    ax.imshow(img, extent=[minx, maxx, miny, maxy], origin='upper')
    ax.set_aspect('equal')
    annotate_map(ax, gdf, bounds)
    plot_gt(ax)
    ax.set_title(title, fontsize=8.5, fontweight='bold')

# Annotation sur la carte diff
axes[4].text(0.02, 0.04,
    'Rouge = zones qui\nréagissent aux pluies\n(sol nu, végétation)\nBleu = béton/routes',
    transform=axes[4].transAxes, fontsize=8, color='white',
    bbox=dict(facecolor='black', alpha=0.7, boxstyle='round'))

plt.tight_layout()
plt.savefig('/outputs/monthly_02_maps_seasonal.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_02_maps_seasonal.png')

# Stats ΔVV par zone (saison pluies - sèche)
print('\nΔVV par moughataa (saison pluies - saison sèche) :')
print(f'{"Zone":20s} {"VV_sèche":>10s} {"VV_pluies":>10s} {"ΔVV":>8s}  Saisonnalité')
print('-'*60)
for _, row in gdf.iterrows():
    zone    = row['ADM2_EN']
    geom_ee = ee.Geometry(_json.loads(gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_json())['features'][0]['geometry'])
    vv_d = comp_dry.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('VV',np.nan)
    vv_w = comp_wet.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('VV',np.nan)
    if vv_d and vv_w:
        diff = float(vv_w)-float(vv_d)
        flag = 'Forte ↑' if diff>0.5 else 'Modérée' if diff>0.2 else 'Faible'
        print(f'{zone:20s} {float(vv_d):>10.2f} {float(vv_w):>10.2f} {diff:>+8.2f}  {flag}')

### 📦 Cellule E — Boxplot Saisonnalité par Mois
Distribution du VV pour chaque mois, poolé sur toutes les années.  
Révèle le **profil saisonnier moyen** de chaque moughataa.

In [ ]:
# ── CELL 11E — Boxplot Saisonnalité Mensuelle ───────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('Saisonnalité du Backscatter VV — Distribution par Mois (2016-2024)\n'
             'Chaque boîte = distribution de ce mois sur toutes les années disponibles',
             fontsize=14, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = df_monthly[df_monthly['zone']==zone].dropna(subset=['vv_mean'])
    color = ZONE_COLORS[zone]

    # Boxplot par mois
    data_by_month = [sub[sub['month']==m]['vv_mean'].values for m in MONTHS]
    bp = ax.boxplot(data_by_month, patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2),
                    whiskerprops=dict(color=color, lw=1.2),
                    capprops=dict(color=color, lw=1.5),
                    flierprops=dict(marker='o', markerfacecolor=color, ms=3, alpha=0.5))
    for patch in bp['boxes']:
        patch.set_facecolor(color); patch.set_alpha(0.35)

    # Profil des médianes
    medians = [np.median(d) if len(d)>0 else np.nan for d in data_by_month]
    ax2 = ax.twinx()
    ax2.plot(range(1, 13), medians, color=color, lw=2.5, marker='o', ms=5, zorder=5)
    ax2.set_ylabel('Médiane VV (dB)', fontsize=7, color=color)
    ax2.tick_params(axis='y', labelcolor=color, labelsize=7)

    # Zones de saison
    ax.axvspan(7, 10, alpha=0.08, color='blue', label='Saison pluies')
    ax.axvspan(1,  4, alpha=0.08, color='orange', label='Saison sèche')

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_NAMES, fontsize=7, rotation=45)
    ax.set_ylabel('VV (dB)', fontsize=8)
    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.grid(alpha=0.2, ls=':')

# Légende commune
axes[0,0].axvspan(0,0, alpha=0.3, color='blue',   label='Saison pluies (Jul-Oct)')
axes[0,0].axvspan(0,0, alpha=0.3, color='orange', label='Saison sèche (Nov-Avr)')
axes[0,0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('/outputs/monthly_03_boxplot_seasonality.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_03_boxplot_seasonality.png')

# Profil moyen ville
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Profil Saisonnier Moyen — Ville de Nouakchott (2016-2024)\n'
             'VV mensuel poolé sur toutes les années', fontsize=12, fontweight='bold')

for zone in ZONE_ORDER:
    sub     = df_monthly[df_monthly['zone']==zone].dropna(subset=['vv_mean'])
    medians = [np.median(sub[sub['month']==m]['vv_mean'].values) if len(sub[sub['month']==m])>0 else np.nan
               for m in MONTHS]
    ax.plot(range(1,13), medians, color=ZONE_COLORS[zone], lw=1.5,
            marker='o', ms=4, alpha=0.7, label=zone)

ax.axvspan(7, 10, alpha=0.08, color='blue',   label='Saison pluies')
ax.axvspan(11,12, alpha=0.06, color='orange')
ax.axvspan(1,  4, alpha=0.06, color='orange', label='Saison sèche')
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTH_NAMES, fontsize=9)
ax.set_ylabel('VV médian (dB)'); ax.set_xlabel('Mois')
ax.legend(fontsize=8, loc='upper right', ncol=2); ax.grid(alpha=0.25, ls=':')
plt.tight_layout()
plt.savefig('/outputs/monthly_03b_seasonal_profile_city.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_03b_seasonal_profile_city.png')

### 📉 Cellule F — Décomposition STL (Tendance + Saisonnier + Résiduel)
Décomposition classique de série temporelle en trois composantes :
- **Tendance** — la croissance urbaine long-terme
- **Saisonnalité** — le cycle annuel répétitif
- **Résiduel** — le bruit restant (bruit SAR, événements rares)

In [ ]:
# ── CELL 11F — Décomposition STL par Moughataa ──────────────────────────────
from statsmodels.tsa.seasonal import STL

fig, axes = plt.subplots(9, 4, figsize=(22, 36))
fig.suptitle('Décomposition STL — VV Mensuel par Moughataa 2016-2024\n'
             'Observed · Trend · Seasonal · Residual',
             fontsize=14, fontweight='bold')

stl_results = {}

for row_i, zone in enumerate(ZONE_ORDER):
    sub = (df_monthly[df_monthly['zone']==zone]
           .sort_values(['year','month'])
           .dropna(subset=['vv_mean'])
           .copy())

    # Créer index mensuel complet
    sub['date'] = pd.to_datetime({'year':sub['year'],'month':sub['month'],'day':1})
    sub = sub.set_index('date')

    # Remplir les éventuels mois manquants par interpolation
    full_idx = pd.date_range(f'{YEARS_MONTHLY[0]}-01-01',
                             f'{YEARS_MONTHLY[-1]}-12-01', freq='MS')
    ts = sub['vv_mean'].reindex(full_idx).interpolate(method='linear')

    # STL : période = 12 mois
    stl   = STL(ts, period=12, robust=True)
    res   = stl.fit()
    stl_results[zone] = res

    color = ZONE_COLORS[zone]
    components = [
        (ts.values,           'Observé (VV brut)', color),
        (res.trend,           'Tendance',          '#1d3557'),
        (res.seasonal,        'Saisonnalité',      '#2a9d8f'),
        (res.resid,           'Résiduel',          '#adb5bd'),
    ]

    for col_i, (data, label, c) in enumerate(components):
        ax = axes[row_i, col_i]
        ax.plot(full_idx, data, color=c, lw=1.5 if col_i>0 else 1.0, alpha=0.9)
        if col_i == 0:
            ax.set_ylabel(zone, fontsize=8, fontweight='bold', color=color)
        if col_i == 2:  # saisonnalité : axhline à 0
            ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
        if col_i == 3:  # résiduel : axhline ±1 dB
            ax.axhline(+1, color='red',  lw=0.8, ls=':', alpha=0.5)
            ax.axhline(-1, color='red',  lw=0.8, ls=':', alpha=0.5)
        if row_i == 0:
            ax.set_title(label, fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.grid(alpha=0.2, ls=':')
        ax.tick_params(labelsize=6)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('/outputs/monthly_04_stl_decomposition.png', dpi=120, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_04_stl_decomposition.png')

# Résumé des amplitudes
print('\nAmplitude saisonnière STL par zone (max-min de la composante saisonnière) :')
print(f'{"Zone":20s} {"Amplitude(dB)":>14s} {"Tendance slope":>16s} (dB/an)')
print('-'*55)
for zone in ZONE_ORDER:
    res   = stl_results[zone]
    amp   = res.seasonal.max() - res.seasonal.min()
    slope = np.polyfit(range(len(res.trend)), res.trend, 1)[0] * 12  # dB/an
    flag  = '↑ urbanisation' if slope > 0.05 else ('↓ ??' if slope < -0.05 else '→ stable')
    print(f'{zone:20s} {amp:>14.3f} {slope:>+16.3f}  {flag}')

### 📊 Cellule G — Test de Kruskal-Wallis (Significativité de la Saisonnalité)
Test non-paramétrique : est-ce que les 12 mois ont des distributions **significativement différentes** ?  
H₀ : toutes les distributions mensuelles sont identiques (pas de saisonnalité).

In [ ]:
# ── CELL 11G — Test Kruskal-Wallis Saisonnalité ─────────────────────────────
from scipy.stats import kruskal, f_oneway

print('Test de Kruskal-Wallis : H0 = pas de différence entre les mois')
print('p < 0.05 → saisonnalité statistiquement significative')
print('='*65)
print(f'{"Zone":20s} {"stat H":>10s} {"p-value":>10s} {"Signif.":>12s} {"Amp.(dB)":>10s}')
print('-'*65)

kw_results = {}
for zone in ZONE_ORDER:
    sub          = df_monthly[df_monthly['zone']==zone].dropna(subset=['vv_mean'])
    groups       = [sub[sub['month']==m]['vv_mean'].values for m in MONTHS]
    groups_clean = [g for g in groups if len(g)>=3]

    if len(groups_clean) < 6:
        print(f'{zone:20s}  {"N/A":>10s} {"N/A":>10s}  données insuffisantes')
        continue

    stat, p = kruskal(*groups_clean)
    amp      = max(np.median(g) for g in groups_clean) - min(np.median(g) for g in groups_clean)
    sig      = '*** p<0.001' if p<0.001 else '**  p<0.01' if p<0.01 else '*   p<0.05' if p<0.05 else 'ns  p>0.05'
    kw_results[zone] = {'stat':stat, 'p':p, 'amp':amp}
    print(f'{zone:20s} {stat:>10.3f} {p:>10.4f} {sig:>12s} {amp:>10.3f}')

print()
print('Interprétation :')
print('  Amplitude > 1.0 dB + p<0.05 → saisonnalité réelle et forte')
print('  Amplitude < 0.5 dB          → saisonnalité dominée par le bruit SAR')
print()

# Figure : p-values et amplitudes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Significativité de la Saisonnalité par Moughataa\n'
             'Test de Kruskal-Wallis sur les 12 distributions mensuelles',
             fontsize=12, fontweight='bold')

zones_kw = list(kw_results.keys())
pvals    = [kw_results[z]['p']   for z in zones_kw]
amps     = [kw_results[z]['amp'] for z in zones_kw]

ax = axes[0]
colors_p = ['#e63946' if p<0.05 else '#adb5bd' for p in pvals]
bars = ax.barh(zones_kw, [-np.log10(p) for p in pvals], color=colors_p, alpha=0.85)
ax.axvline(-np.log10(0.05), color='black', lw=1.5, ls='--', label='p=0.05')
ax.axvline(-np.log10(0.01), color='grey',  lw=1,   ls=':',  label='p=0.01')
ax.set_xlabel('-log10(p-value)  [plus grand = plus significatif]')
ax.set_title('Significativité (Kruskal-Wallis)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3, ls=':')

ax = axes[1]
colors_a = ['#2a9d8f' if a>0.5 else '#adb5bd' for a in amps]
ax.barh(zones_kw, amps, color=colors_a, alpha=0.85)
ax.axvline(0.5, color='black', lw=1.5, ls='--', label='Seuil 0.5 dB')
ax.axvline(1.0, color='grey',  lw=1,   ls=':',  label='Seuil 1.0 dB')
ax.set_xlabel('Amplitude saisonnière (dB)  [médiane max - médiane min]')
ax.set_title('Amplitude Saisonnière', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3, ls=':')

plt.tight_layout()
plt.savefig('/outputs/monthly_04b_kruskal_wallis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_04b_kruskal_wallis.png')

### 🗺️ Cellule H — Carte Différence Saisonnière Spatiale
Choroplèthe par moughataa : où la saisonnalité est-elle la plus forte ?  
**Hypothèse :** Les zones les plus urbanisées (béton imperméable) auront la **plus faible** amplitude — inverse logique de la carte d'urbanisation.

In [ ]:
# ── CELL 11H — Carte Choroplèthe Amplitude Saisonnière ──────────────────────
# Thumbnail de la différence VV août - janvier sur toute la ville

print('Téléchargement carte différence saisonnière...')
vv_aug_full = s1_col.filter(ee.Filter.calendarRange(8,8,'month')).select('VV').median()
vv_jan_full = s1_col.filter(ee.Filter.calendarRange(1,1,'month')).select('VV').median()
vv_aug_jan  = vv_aug_full.subtract(vv_jan_full).rename('seasonal_diff')

vv_aug_thumb2 = get_colored_thumb(vv_aug_full,'VV',region_ee,PALETTE_VV,-20,-8)
vv_jan_thumb2 = get_colored_thumb(vv_jan_full,'VV',region_ee,PALETTE_VV,-20,-8)
vv_diff_thumb2 = get_colored_thumb(vv_aug_jan,'seasonal_diff',region_ee,
                    ['2166ac','92c5de','f7f7f7','f4a582','d6604d'],-2,2)
print('✓ OK')

fig = plt.figure(figsize=(22, 12))
fig.suptitle('Saisonnalité Spatiale SAR — Amplitude Août vs Janvier\n'
             'Nouakchott 2016-2024 (composites multi-annuels)',
             fontsize=13, fontweight='bold')

gs = gridspec.GridSpec(1, 4, figure=fig, wspace=0.05)

# 3 cartes SAR
for col_i, (img, title) in enumerate([
    (vv_jan_thumb2,  'VV Janvier\n(saison sèche)'),
    (vv_aug_thumb2,  'VV Août\n(saison des pluies)'),
    (vv_diff_thumb2, 'ΔVV Août−Janvier\nRouge=+dB pluies | Bleu=−dB'),
]):
    ax = fig.add_subplot(gs[0, col_i])
    ax.imshow(img, extent=[minx,maxx,miny,maxy], origin='upper')
    ax.set_aspect('equal')
    annotate_map(ax, gdf, bounds)
    plot_gt(ax)
    ax.set_title(title, fontsize=9, fontweight='bold')

# Choroplèthe amplitude KW par moughataa
ax4 = fig.add_subplot(gs[0, 3])
gdf_seasonal = gdf.copy()
gdf_seasonal['amplitude'] = gdf_seasonal['ADM2_EN'].map(
    {z: kw_results[z]['amp'] if z in kw_results else np.nan for z in ZONE_ORDER}
)
gdf_seasonal['signif'] = gdf_seasonal['ADM2_EN'].map(
    {z: kw_results[z]['p']<0.05 if z in kw_results else False for z in ZONE_ORDER}
)

cmap_amp = LinearSegmentedColormap.from_list('amplitude',['#1d3557','#a8dadc','#e63946'])
gdf_seasonal.plot(column='amplitude', ax=ax4, cmap=cmap_amp,
                  vmin=0, vmax=gdf_seasonal['amplitude'].max(),
                  edgecolor='white', linewidth=1.5,
                  missing_kwds={'color':'lightgrey'})

for _, row in gdf_seasonal.iterrows():
    amp  = row['amplitude']
    sig  = row['signif']
    star = '*' if sig else ''
    if not np.isnan(amp):
        ax4.annotate(f"{row['ADM2_EN']}
{amp:.2f}dB{star}",
                     (row.geometry.centroid.x, row.geometry.centroid.y),
                     fontsize=7.5, ha='center', va='center', fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2',facecolor='white',alpha=0.75,linewidth=0))

sm = plt.cm.ScalarMappable(cmap=cmap_amp,
     norm=plt.Normalize(vmin=0, vmax=gdf_seasonal['amplitude'].max()))
sm.set_array([])
plt.colorbar(sm, ax=ax4, shrink=0.6, label='Amplitude saisonnière (dB)')
ax4.set_title('Amplitude Saisonnière par Moughataa\n(* = p<0.05 Kruskal-Wallis)',
              fontsize=9, fontweight='bold')
ax4.set_xticks([]); ax4.set_yticks([])

plt.savefig('/outputs/monthly_05_map_diff_saisons.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_05_map_diff_saisons.png')
print()
print('Corrélation amplitude saisonnière vs fraction urbaine 2024 :')
amps_ordered = [kw_results[z]['amp'] if z in kw_results else np.nan for z in ZONE_ORDER]
ufs_ordered  = [uf_2024[z] for z in ZONE_ORDER]
mask_corr    = ~np.isnan(amps_ordered)
from scipy.stats import pearsonr
r_amp, p_amp = pearsonr(np.array(amps_ordered)[mask_corr], np.array(ufs_ordered)[mask_corr])
print(f'  r = {r_amp:.3f}  p = {p_amp:.4f}')
print(f'  → {"Corrélation négative ✓ — plus urbanisé = moins saisonnier" if r_amp<0 else "Résultat inattendu"}')

### 📈 Cellule I — Synthèse Finale : Série Mensuelle Lissée
Figure publication-ready : série mensuelle brute + tendance lissée (moyenne mobile 12 mois) pour chaque moughataa et pour la ville.

In [ ]:
# ── CELL 11I — Synthèse Finale : Série Mensuelle + Tendance Lissée ──────────
fig, axes = plt.subplots(3, 3, figsize=(22, 16))
fig.suptitle('Série Mensuelle VV + Tendance Lissée (MM-12) — Nouakchott 2016-2024\n'
             'Gris clair = VV brut mensuel  |  Couleur = moyenne mobile 12 mois  |  '
             'Zone bleue = saison des pluies (Jul-Oct)',
             fontsize=13, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = (df_monthly[df_monthly['zone']==zone]
             .sort_values(['year','month'])
             .dropna(subset=['vv_mean'])
             .copy())
    sub['date']   = pd.to_datetime({'year':sub['year'],'month':sub['month'],'day':15})
    sub['mm12']   = sub['vv_mean'].rolling(window=12, center=True, min_periods=6).mean()
    color         = ZONE_COLORS[zone]

    # Zones pluies (juil-oct) sur toutes les années
    for year in YEARS_MONTHLY:
        ax.axvspan(pd.Timestamp(f'{year}-07-01'), pd.Timestamp(f'{year}-10-31'),
                   alpha=0.06, color='#457b9d', linewidth=0)

    # VV brut mensuel
    ax.plot(sub['date'], sub['vv_mean'], color='#cccccc', lw=0.8, alpha=0.7, zorder=1)

    # Moyenne mobile 12 mois
    ax.plot(sub['date'], sub['mm12'], color=color, lw=2.5, zorder=3,
            label='MM-12')

    # Tendance STL si disponible
    if zone in stl_results:
        full_idx = pd.date_range(f'{YEARS_MONTHLY[0]}-01-01',
                                 f'{YEARS_MONTHLY[-1]}-12-01', freq='MS')
        ax.plot(full_idx, stl_results[zone].trend,
                color=color, lw=1.5, ls='--', alpha=0.6, zorder=2,
                label='Tendance STL')

    # Annotation amplitude et tendance
    if zone in kw_results:
        amp  = kw_results[zone]['amp']
        p_kw = kw_results[zone]['p']
        sig  = '✓ sig.' if p_kw<0.05 else 'ns'
        ax.text(0.02, 0.05,
                f'Amp. sais. = {amp:.2f} dB ({sig})',
                transform=ax.transAxes, fontsize=7.5, color=color,
                bbox=dict(facecolor='white', alpha=0.75, boxstyle='round', linewidth=0))

    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('VV (dB)', fontsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2, ls=':')
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

plt.tight_layout()
plt.savefig('/outputs/monthly_06_final_synthesis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_06_final_synthesis.png')

# ── Figure ville globale ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
fig.suptitle('Indice VV Mensuel — Ville de Nouakchott (pondéré superficie) 2016-2024\n'
             'Gris = brut mensuel  |  Bleu foncé = MM-12  |  Rouge = tendance STL',
             fontsize=13, fontweight='bold')

df_city_monthly_sorted = df_city_monthly.sort_values(['year','month']).copy()
df_city_monthly_sorted['date'] = pd.to_datetime(
    {'year':df_city_monthly_sorted['year'],
     'month':df_city_monthly_sorted['month'],'day':15})
df_city_monthly_sorted['mm12'] = (df_city_monthly_sorted['vv_city']
                                   .rolling(window=12, center=True, min_periods=6).mean())

for year in YEARS_MONTHLY:
    ax.axvspan(pd.Timestamp(f'{year}-07-01'), pd.Timestamp(f'{year}-10-31'),
               alpha=0.08, color='#457b9d', linewidth=0)

ax.plot(df_city_monthly_sorted['date'], df_city_monthly_sorted['vv_city'],
        color='#cccccc', lw=0.8, alpha=0.8, label='VV brut mensuel')
ax.plot(df_city_monthly_sorted['date'], df_city_monthly_sorted['mm12'],
        color='#1d3557', lw=3, label='Moyenne mobile 12 mois')

# STL ville
sub_city   = df_city_monthly_sorted.dropna(subset=['vv_city'])
full_idx_c = pd.date_range(f'{YEARS_MONTHLY[0]}-01-01',
                            f'{YEARS_MONTHLY[-1]}-12-01', freq='MS')
ts_city    = sub_city.set_index('date')['vv_city'].reindex(full_idx_c).interpolate()
stl_city   = STL(ts_city, period=12, robust=True).fit()
ax.plot(full_idx_c, stl_city.trend, color='#e63946', lw=2, ls='--',
        alpha=0.9, label='Tendance STL')

slope_city = np.polyfit(range(len(stl_city.trend)), stl_city.trend, 1)[0] * 12
ax.set_ylabel('VV moyen pondéré (dB)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.25, ls=':')
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax.tick_params(axis='x', labelsize=9)

ax.text(0.98, 0.05,
        f'Tendance STL ville : {slope_city:+.3f} dB/an',
        transform=ax.transAxes, ha='right', fontsize=10, color='#e63946',
        fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.8, boxstyle='round', linewidth=0))

plt.tight_layout()
plt.savefig('/outputs/monthly_06b_city_global_monthly.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_06b_city_global_monthly.png')
print()
print('=== RÉCAPITULATIF PARTIE 11 ===')
print(f'  Fichiers produits :')
for f in ['monthly_01_heatmap.png', 'monthly_01b_heatmap_city.png',
          'monthly_02_maps_seasonal.png', 'monthly_03_boxplot_seasonality.png',
          'monthly_03b_seasonal_profile_city.png', 'monthly_04_stl_decomposition.png',
          'monthly_04b_kruskal_wallis.png', 'monthly_05_map_diff_saisons.png',
          'monthly_06_final_synthesis.png', 'monthly_06b_city_global_monthly.png',
          'monthly_sar_vv.csv']:
    print(f'  ✓ /outputs/{f}')